# 👁️ La pupila como índice del **estado poblacional de VISp**
### `04_pupila_estado_VISp` · NeuroDataReHack 2026 · Allen Visual Coding – Neuropixels (DANDI:000021)

**Pregunta científica única:** ¿en qué medida la **pupila** indexa el **estado poblacional** de la
corteza visual primaria (VISp) del ratón durante la **actividad espontánea**, y con qué **latencia
temporal**?

Este notebook responde esa pregunta con un pipeline corto y honesto, en el orden en que importa:

- **Bloque 0 — Viabilidad (innegociable):** ¿hay suficiente espontáneo y suficientes neuronas limpias?
- **Bloque 1 — Sustrato común:** alineamos pupila, carrera y spikes de VISp en una rejilla común y
  extraemos el **estado neuronal `Y`** con PCA (ajustado *solo* con el tramo de entrenamiento).
- **Bloque 2 — Correlación cruzada + nulo circular (el cimiento):** ¿están acopladas pupila y estado,
  con qué retardo, y es significativo frente a un nulo que respeta la autocorrelación? *Nunca se corta.*
- **Bloque 3 — *Decoding* con barrido de lag (refuerzo):** reconstruir el estado desde la pupila con un
  **GLM** y una **GRU**, barriendo el retardo τ. El τ óptimo debería coincidir con el pico del Bloque 2.
- **Bloque 4 — Contraste (opcional):** *forecasting* intrínseco del estado (sin pupila).
- **Bloque 5 — Relato y límites declarados.**

> **Por qué esta pregunta y este régimen.** Trabajamos en **actividad espontánea** (pantalla gris,
> luminancia constante). Con estímulo visual, VISp está dominada por respuestas evocadas y "arousal" se
> confunde con "respuesta al estímulo"; además una luminancia cambiante rompería el supuesto
> *pupila = arousal*. La luminancia constante es justo lo que hace interpretable la pupila.

> **Nota de rendimiento.** El NWB de sesión pesa varios GB, pero **no lo descargamos**: usamos
> *streaming* con `remfile` y leemos de la nube solo lo que pedimos (metadatos y los `spike_times` de
> las unidades seleccionadas).

---
## Bloque 0 — Entorno, imports y apertura del NWB

Empezamos importando las librerías. El conjunto es deliberadamente **reducido** (decisión del SPEC,
§2 y §5): núcleo científico + *streaming* NWB + `pynapple` para alinear series temporales +
`scikit-learn` para PCA/estandarización/R² + `scipy.stats` para las correlaciones + `torch` para la
GRU. **No** usamos `tsfel`, `xgboost` ni `nemos`: el objetivo (el estado `Y`) es continuo, así que un
GLM *gaussiano* es literalmente una **regresión lineal regularizada** (`Ridge`), y las *features* de
pupila que necesitamos son solo dos (la pupila suavizada y su derivada). Menos piezas = menos frágil.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

# --- Nucleo numerico y graficos ---
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# --- Streaming NWB / DANDI (leer el archivo remoto sin descargarlo) ---
import h5py
import remfile
import pynwb
from dandi.dandiapi import DandiAPIClient

# --- Series temporales de neurociencia: alineacion en rejilla comun ---
import pynapple as nap

# --- ML clasico: estandarizar, PCA, GLM lineal (Ridge) y metrica R2 ---
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score

# --- Estadistica: rangos (Spearman) y suavizado gaussiano ---
from scipy.stats import rankdata
from scipy.ndimage import gaussian_filter1d

# --- Deep learning: la GRU del Bloque 3 ---
import torch
import torch.nn as nn

print("Versiones del entorno:")
for nombre, mod in [("numpy", np), ("pandas", pd), ("pynwb", pynwb),
                    ("pynapple", nap), ("torch", torch)]:
    print(f"  {nombre:10s} {mod.__version__}")

### 0.1 ¿Hay GPU? (acelera la GRU, pero no es obligatoria)

`torch.cuda.is_available()` nos dice si hay una GPU NVIDIA utilizable. Creamos un objeto `DEVICE` que
le indica a `torch` dónde colocar los tensores y el modelo. Si no hay GPU, todo funciona igual en CPU
(solo algo más lento). Fijamos la **semilla** para que los resultados sean reproducibles.

In [ ]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if DEVICE.type == "cuda":
    print(f"GPU detectada: {torch.cuda.get_device_name(0)}  (CUDA {torch.version.cuda})")
else:
    print("Sin GPU: se usara CPU (el pipeline funciona igual, algo mas lento).")
print("Dispositivo de computo:", DEVICE)

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
rng = np.random.default_rng(SEED)   # generador moderno de numpy (para el nulo del Bloque 2)

### 0.2 Configuración central (una sola celda)

Reunimos **todos** los parámetros ajustables aquí (SPEC §3). Cambia un valor y re-ejecuta desde este
punto. Los umbrales de QC son los valores por defecto del Allen Institute; el bin de 50 ms + suavizado
de 100 ms son estándar para análisis de poblaciones (un bin de 50 ms de una sola neurona es casi
siempre 0/1 spike = ruido; el acoplamiento de estado vive en cientos de ms a segundos).

In [ ]:
# --- Que datos cargar ---
DANDISET_ID    = "000021"
DANDI_FILEPATH = "sub-738651046/sub-738651046_ses-760693773.nwb"
REGION         = "VISp"          # region objetivo (acronimo del atlas Allen CCF)

# --- Rejilla temporal ---
BIN          = 0.050             # s  (50 ms) tamano de bin
SMOOTH_SIGMA = 0.100             # s  (100 ms) sigma del suavizado gaussiano de tasas

# --- Control de calidad del spike-sorting (umbrales Allen) ---
QC_QUALITY        = "good"
QC_ISI_MAX        = 0.5
QC_AMP_CUTOFF_MAX = 0.1
QC_PRESENCE_MIN   = 0.9

# --- Estado neuronal (PCA) ---
N_PCA      = 3                   # nº de componentes del estado neuronal Y
FRAC_TRAIN = 0.70                # split temporal secuencial, SIN barajar

# --- Bloque 2: correlacion cruzada + nulo ---
LAG_MAX     = 3.0                # s  rango de lags de la correlacion cruzada: +/- LAG_MAX
N_PERM      = 2000               # permutaciones circulares para el nulo
CORR_METHOD = "spearman"         # "spearman" (recomendado por no-gaussianidad) o "pearson"
SENAL_NEURAL = "PC1"             # "PC1" (por defecto) o "mean_z" (media de z-scores)

# --- Bloque 3: decoding con barrido de lag tau ---
TAU_GRID = np.round(np.arange(-2.0, 2.0 + 1e-9, 0.25), 3)   # s  (de -2 a +2 en pasos de 0.25 s)
DEC_SEQ  = 20                    # nº de bins de pupila reciente que ve la GRU (20 x 50 ms = 1 s)
RUN_GRU  = True                  # False = recorte del SPEC: dejar solo el GLM
DETREND_STATE = False            # True = quitar la deriva lineal (train-fit) de Y y features antes del decoding

# --- GRU ---
GRU_HIDDEN = 32
GRU_EPOCHS = 80
GRU_LR     = 5e-3
GRU_BATCH  = 256

# --- Bloque 4 (opcional; primero en eliminarse si falta tiempo) ---
RUN_BLOQUE4  = True
AR_LAGS      = 10                # bins de historia del estado como predictores
AR_HORIZONTES = [1, 5, 10]       # horizontes de forecasting (bins): 50, 250, 500 ms

print("Configuracion cargada.")
print(f"  Region: {REGION} | bin: {BIN*1000:.0f} ms | suavizado: {SMOOTH_SIGMA*1000:.0f} ms")
print(f"  LAG_MAX: +/-{LAG_MAX:.1f} s | N_PERM: {N_PERM} | TAU_GRID: {TAU_GRID[0]}..{TAU_GRID[-1]} s")

### 0.3 Abrir el NWB por *streaming*

El `DandiAPIClient` nos da la **URL S3** del archivo. Con `remfile` la abrimos como si fuera local,
la envolvemos con `h5py` y la interpreta `pynwb`. La primera lectura tarda unos segundos (lee la
*estructura*), pero los datos pesados se leen de forma **perezosa**: solo cuando los pedimos.

In [ ]:
with DandiAPIClient() as client:
    dandiset = client.get_dandiset(DANDISET_ID, "draft")
    asset    = dandiset.get_asset_by_path(DANDI_FILEPATH)
    s3_url   = asset.get_content_url(follow_redirects=1, strip_query=True)

rem_file = remfile.File(s3_url)
h5_file  = h5py.File(rem_file, "r")
io       = pynwb.NWBHDF5IO(file=h5_file, load_namespaces=True)
nwb      = io.read()

print("Sesion    :", nwb.session_id)
print("Sujeto    :", nwb.subject.subject_id, "|", nwb.subject.genotype)
print("Unidades  :", len(nwb.units.id[:]))
print("Intervals :", list(nwb.intervals.keys()))

---
## Bloque 0 (continuación) — **Validación de viabilidad** (PRIMERO, innegociable)

Todo el proyecto asume dos cosas: (1) **suficiente actividad espontánea** y (2) **suficientes neuronas
limpias de VISp**. Antes de invertir tiempo, lo comprobamos. Esta celda imprime:

1. La **duración total del bloque espontáneo** (sumando `stop_time − start_time` de
   `spontaneous_presentations`), en minutos.
2. El **nº de unidades de VISp** que superan el QC.

Y emite un **veredicto**:
- **PLAN A** (seguir tal cual) si hay ≳ 4–5 min de espontáneo **y** ≳ 30 unidades VISp tras QC.
- **PLAN B** si no: incluir además los bloques `natural_movie_*` (se repiten → más minutos),
  **anotando** que introducen componente visual y debilitan la lectura *pupila = arousal*.

In [ ]:
# --- (1) Duracion del bloque espontaneo ---
spont = nwb.intervals["spontaneous_presentations"].to_dataframe()
dur_spont_s   = float((spont["stop_time"] - spont["start_time"]).sum())
dur_spont_min = dur_spont_s / 60.0
print(f"Bloque espontaneo: {len(spont)} intervalos | duracion total = {dur_spont_min:.2f} min")

# --- (2) Unidades de VISp que superan el QC ---
colnames = list(nwb.units.colnames)
cols_meta = [c for c in ["peak_channel_id", "quality", "isi_violations",
                         "amplitude_cutoff", "presence_ratio", "firing_rate", "snr"]
             if c in colnames]
units_meta = pd.DataFrame({c: nwb.units[c].data[:] for c in cols_meta},
                          index=pd.Index(nwb.units.id[:], name="unit_id"))
electrodos = nwb.electrodes.to_dataframe()
units_meta["location"] = units_meta["peak_channel_id"].map(electrodos["location"])
if "quality" in units_meta.columns:
    units_meta["quality"] = units_meta["quality"].astype(str)

# Mascara de QC (aplicamos cada filtro solo si su columna existe -> robusto).
mask = units_meta["location"] == REGION
if "quality"        in units_meta: mask &= units_meta["quality"] == QC_QUALITY
if "isi_violations" in units_meta: mask &= units_meta["isi_violations"] <= QC_ISI_MAX
if "amplitude_cutoff" in units_meta: mask &= units_meta["amplitude_cutoff"] <= QC_AMP_CUTOFF_MAX
if "presence_ratio" in units_meta: mask &= units_meta["presence_ratio"] >= QC_PRESENCE_MIN

n_visp_qc = int(mask.sum())
print(f"Unidades en {REGION} tras QC: {n_visp_qc}")

# --- Veredicto ---
ok_tiempo   = dur_spont_min >= 4.0
ok_neuronas = n_visp_qc     >= 30
print("\n" + "="*56)
if ok_tiempo and ok_neuronas:
    print("VEREDICTO: PLAN A -> hay espontaneo y neuronas suficientes.")
else:
    print("VEREDICTO: PLAN B -> considerar anadir bloques natural_movie_*.")
    print("  (introducen componente visual: la lectura 'pupila=arousal' se debilita ahi).")
print(f"  espontaneo >= 4-5 min: {ok_tiempo}  ({dur_spont_min:.2f} min)")
print(f"  unidades VISp >= ~30 : {ok_neuronas}  ({n_visp_qc} unidades)")
print("="*56)

---
## Bloque 1 — Sustrato común: alineación pupila ↔ carrera ↔ spikes

Aquí construimos la materia prima de todo lo demás. Los pasos y su *porqué*:

- **Rejilla común:** sin el mismo reloj y la misma resolución no hay comparación posible entre pupila
  (a ~30 Hz) y spikes (eventos puntuales). Binamos todo a 50 ms.
- **Suavizado ~100 ms:** reduce el ruido 0/1 de cada neurona y deja ver el acoplamiento de estado.
- **PCA:** la pupila es **unidimensional**; no tiene sentido correlacionarla contra ~90 neuronas
  sueltas. PC1 suele capturar la modulación **global** de estado.
- **Carrera:** *arousal* y locomoción son disociables y la carrera domina la señal; sin registrarla no
  se puede distinguir "pupila indexa cortex" de "ambas siguen a la carrera".
- **Fit *train-only*:** el scaler y el PCA se ajustan **solo** con el primer 70 % de bins → la
  definición del estado no usa información del futuro (evita *leakage*).

### 1.1 Seleccionar unidades VISp + QC (sin cargar aún los `spike_times`)

In [ ]:
unidades_sel   = units_meta[mask]
posiciones_sel = np.where(mask.values)[0]   # indices de fila para leer spike_times despues
print(f"Unidades seleccionadas en {REGION}: {len(unidades_sel)}")
display(unidades_sel[[c for c in ['firing_rate','snr','isi_violations',
                                  'amplitude_cutoff','presence_ratio'] if c in unidades_sel]].head())

### 1.2 `IntervalSet` con **todo** el bloque espontáneo

A diferencia del notebook previo (que usaba 5 s de *drifting gratings*), aquí el dominio temporal es el
**bloque espontáneo completo**: construimos un `IntervalSet` con **todos** sus intervalos. `pynapple`
bineará dentro de cada intervalo y los concatenará, de modo que solo entra tiempo espontáneo.

In [ ]:
ep = nap.IntervalSet(start=spont["start_time"].values, end=spont["stop_time"].values)
print(f"IntervalSet espontaneo: {len(ep)} intervalos | {ep.tot_length()/60:.2f} min en total")

### 1.3 Cargar spikes (solo unidades seleccionadas) → `TsGroup`

Leemos `spike_times` **solo** de las neuronas seleccionadas (lectura perezosa desde la nube) y las
metemos en un `nap.TsGroup`. Al binear con `ep`, `pynapple` recorta automáticamente al espontáneo.

In [ ]:
spikes_dict = {}
for uid, pos in zip(unidades_sel.index, posiciones_sel):
    st = np.asarray(nwb.units["spike_times"][pos])          # todos los spikes de esa neurona
    st = st[(st >= float(ep.start[0])) & (st <= float(ep.end[-1]))]   # recorte grueso a la ventana
    spikes_dict[int(uid)] = nap.Ts(t=st)
tsgroup = nap.TsGroup(spikes_dict)
print(f"TsGroup: {len(tsgroup)} neuronas cargadas.")

### 1.4 Cargar la pupila e interpolar sus NaN → `Tsd`

La pupila (`filtered_gaze_mapping/pupil_area`) contiene **NaN** por parpadeos. Los rellenamos por
interpolación lineal *antes* de binear, para no propagar huecos. El resultado es un `nap.Tsd`.

In [ ]:
pupil_obj = nwb.processing["filtered_gaze_mapping"]["pupil_area"]
pup_val   = np.asarray(pupil_obj.data[:], dtype=float)
pup_time  = np.asarray(pupil_obj.timestamps[:], dtype=float)
nan_mask  = np.isnan(pup_val)
pup_val[nan_mask] = np.interp(pup_time[nan_mask], pup_time[~nan_mask], pup_val[~nan_mask])
pupila = nap.Tsd(t=pup_time, d=pup_val)
print(f"Pupila: {len(pup_val)} muestras | NaN interpolados: {int(nan_mask.sum())} "
      f"({100*nan_mask.mean():.1f} %)")

### 1.5 Localizar la **velocidad de carrera** (ubicación variable → escaneo)

La señal de carrera **cambia de sitio entre versiones** del dataset. En lugar de asumir una ruta,
**escaneamos** `nwb.processing` y `nwb.acquisition` buscando nombres que contengan `run`, `speed` o
`velocity`, imprimimos los candidatos y elegimos el primero. Si el automático fallara, edita
`RUN_KEY_MANUAL` con la ruta correcta (formato `"contenedor/senal"` o `"acquisition/senal"`).

In [ ]:
RUN_KEY_MANUAL = None   # <-- EDITA aqui si el escaneo automatico no acierta, p.ej. "running/running_speed"

def escanear_carrera(nwb):
    cand = []
    for mod_name, mod in nwb.processing.items():          # processing modules
        for name in mod.data_interfaces.keys():
            if any(k in name.lower() for k in ("run", "speed", "velocity")):
                cand.append(("processing", f"{mod_name}/{name}"))
    for name in nwb.acquisition.keys():                   # acquisition
        if any(k in name.lower() for k in ("run", "speed", "velocity")):
            cand.append(("acquisition", name))
    return cand

candidatos = escanear_carrera(nwb)
print("Candidatos de senal de carrera encontrados:")
for origen, ruta in candidatos:
    print(f"  [{origen}] {ruta}")

def cargar_ts_generico(nwb, origen, ruta):
    if origen == "processing":
        mod, name = ruta.split("/", 1)
        obj = nwb.processing[mod][name]
    else:
        obj = nwb.acquisition[ruta]
    d = np.asarray(obj.data[:], dtype=float)
    if getattr(obj, "timestamps", None) is not None:
        t = np.asarray(obj.timestamps[:], dtype=float)
    else:                                                  # reconstruir tiempos desde rate
        t0 = float(getattr(obj, "starting_time", 0.0) or 0.0)
        rate = float(obj.rate)
        t = t0 + np.arange(len(d)) / rate
    return t, d

if RUN_KEY_MANUAL is not None:
    origen = "acquisition" if RUN_KEY_MANUAL in nwb.acquisition else "processing"
    run_t, run_d = cargar_ts_generico(nwb, origen, RUN_KEY_MANUAL)
    print(f"\nCarrera (manual): {RUN_KEY_MANUAL}")
elif candidatos:
    origen, ruta = candidatos[0]
    run_t, run_d = cargar_ts_generico(nwb, origen, ruta)
    print(f"\nCarrera (auto): [{origen}] {ruta}")
else:
    run_t = run_d = None
    print("\nAVISO: no se encontro senal de carrera. Se continua SIN carrera (revisar arriba).")

if run_d is not None:
    run_d = np.abs(run_d)                                  # velocidad = magnitud (evita signo del encoder)
    if np.isnan(run_d).any():
        ok = ~np.isnan(run_d)
        run_d[~ok] = np.interp(run_t[~ok], run_t[ok], run_d[ok])
    carrera = nap.Tsd(t=run_t, d=run_d)
    print(f"Carrera: {len(run_d)} muestras | rango [{run_d.min():.2f}, {run_d.max():.2f}]")

### 1.6 Binear todo a la MISMA rejilla y alinear longitudes

Convertimos spikes→tasas suavizadas, y pupila/carrera→media por bin, todo sobre `ep`. Luego recortamos
al mínimo número de bins común y **verificamos con un `assert`** que los tiempos coinciden.

In [ ]:
# (1) Spikes -> tasas suavizadas
conteos = tsgroup.count(BIN, ep=ep)          # TsdFrame (bins x neuronas)
tasas   = conteos / BIN                        # Hz
tasas_s = tasas.smooth(SMOOTH_SIGMA)           # suavizado gaussiano (sigma en s)

# (2) Pupila y carrera -> media por bin en la misma rejilla
pupila_b = pupila.bin_average(BIN, ep=ep)
tiene_carrera = ("carrera" in dir())
if tiene_carrera:
    carrera_b = carrera.bin_average(BIN, ep=ep)

# (3) Alineacion robusta: mismo nº de bins + verificacion de tiempos
t_rate = np.asarray(tasas_s.t); t_pup = np.asarray(pupila_b.t)
L = min(len(t_rate), len(t_pup))
if tiene_carrera:
    L = min(L, len(np.asarray(carrera_b.t)))
tiempos = t_rate[:L]
RATES   = np.asarray(tasas_s.values)[:L]                   # (n_bins, n_neuronas)
PUPILA  = np.asarray(pupila_b.values)[:L].astype(float)    # (n_bins,)
CARRERA = np.asarray(carrera_b.values)[:L].astype(float) if tiene_carrera else None

# Interpolar bins vacios (NaN) sobre el indice de bin
def interp_nan(v):
    if v is None or not np.isnan(v).any():
        return v
    idx = np.arange(len(v)); ok = ~np.isnan(v)
    v[~ok] = np.interp(idx[~ok], idx[ok], v[ok]); return v
PUPILA = interp_nan(PUPILA); CARRERA = interp_nan(CARRERA)

assert np.max(np.abs(t_rate[:L] - t_pup[:L])) < BIN, "Los tiempos NO estan alineados"
n_bins, n_neuronas = RATES.shape
print(f"RATES: {RATES.shape} (bins x neuronas) | PUPILA: {PUPILA.shape} | "
      f"carrera: {'si' if tiene_carrera else 'NO'}")
print(f"Base temporal comun verificada: {n_bins} bins de {BIN*1000:.0f} ms "
      f"({n_bins*BIN/60:.2f} min)")

### 1.7 Estado neuronal `Y` con PCA **train-only**

Split temporal secuencial (sin barajar): primer 70 % = *train*, último 30 % = *test*. Estandarizamos
cada neurona y ajustamos el PCA **solo con train**; luego proyectamos todo. `Y` tiene forma
`(n_bins × N_PCA)`.

In [ ]:
cut = int(FRAC_TRAIN * n_bins)
print(f"Train: bins [0, {cut})  |  Test: bins [{cut}, {n_bins})")

scaler_rates = StandardScaler().fit(RATES[:cut])
RATES_z = scaler_rates.transform(RATES)

pca = PCA(n_components=N_PCA).fit(RATES_z[:cut])
Y   = pca.transform(RATES_z)                                # estado neuronal (n_bins, N_PCA)
evr = pca.explained_variance_ratio_
print(f"Estado neuronal Y: {Y.shape}")
print(f"Varianza explicada PC1-{N_PCA}: {np.round(evr*100,1)} %  (acumulada {evr.sum()*100:.1f} %)")

### 1.8 Figura de comprobación: raster + pupila alineada

La mejor forma de confiar en la alineación es *verla*. Dibujamos, compartiendo el eje X (una porción de
~60 s), el **raster** de spikes arriba y la **pupila** (con la carrera, si existe) debajo. Deberían
verse subidas de actividad poblacional acompañadas —con cierto retardo— de subidas de pupila.

In [ ]:
t0 = float(tiempos[0]); t1 = t0 + 60.0                     # primeros ~60 s de espontaneo
fig, (axr, axp) = plt.subplots(2, 1, figsize=(13, 7), sharex=True,
                               gridspec_kw={"height_ratios": [3, 1]})

# Raster: solo spikes de la porcion mostrada
for i, uid in enumerate(unidades_sel.index):
    st = np.asarray(tsgroup[int(uid)].t)
    st = st[(st >= t0) & (st <= t1)]
    axr.vlines(st, i + 0.5, i + 1.5, color="black", lw=0.4)
axr.set_ylabel("Neurona (idx)"); axr.set_ylim(0.5, len(unidades_sel) + 0.5)
axr.set_title(f"Alineacion: raster VISp ({len(unidades_sel)} neuronas) + pupila / carrera")

m = (tiempos >= t0) & (tiempos <= t1)
axp.plot(tiempos[m], PUPILA[m], color="tab:purple", lw=1.2, label="pupila (area)")
if CARRERA is not None:
    axc = axp.twinx()
    axc.plot(tiempos[m], CARRERA[m], color="tab:green", lw=1.0, alpha=0.7, label="carrera")
    axc.set_ylabel("carrera", color="tab:green")
axp.set_xlabel("Tiempo (s)"); axp.set_ylabel("pupila", color="tab:purple")
axp.set_xlim(t0, t1)
plt.tight_layout(); plt.show()

---
## Bloque 2 — Correlación cruzada con lag + control nulo (EL CIMIENTO)

Este bloque responde la pregunta por sí solo, y por eso **nunca se corta**. Dos ideas:

**1) Correlación cruzada, no simple.** La pupila va **retrasada** respecto a la actividad neuronal
(vía periférica lenta): un análisis a lag cero perdería o infravaloraría la relación. Calculamos la
correlación entre la señal neuronal 1D (**PC1** por defecto) y la pupila para cada retardo en
`±LAG_MAX`, y reportamos el **lag del pico**. Como el **signo de PC1 es arbitrario** (PCA puede
devolver PC1 o −PC1), el estadístico es el **máximo de |correlación|**: nos importa la *magnitud* del
acoplamiento, no su signo. Convención: lag **positivo** = la pupila va **por detrás** del estado.

**2) Nulo por permutación circular (innegociable).** Ambas señales son lentas y muy
autocorrelacionadas; un p-valor ingenuo (que asume independencia entre muestras) daría significancia
espuria a *cualquier* par de señales lentas. La solución: desplazar **cíclicamente** una señal respecto
a la otra un *offset* aleatorio y recalcular la correlación máxima, muchas veces. El desplazamiento
circular **preserva la autocorrelación** de cada señal y rompe **solo** la relación temporal entre
ambas → nulo correcto.

> **Implementación eficiente y equivalente.** Desplazar `np.roll` + recomputar la ventana de lags es,
> matemáticamente, tomar una **ventana de la función de correlación cruzada circular** centrada en el
> *offset*. La calculamos **una vez** con FFT y luego cada permutación es solo un `max` sobre una
> ventana pequeña — mismo nulo, muchísimo más rápido. Con Spearman, correlacionamos los **rangos**
> (Spearman = Pearson sobre rangos), así que rankeamos una vez y todo lo demás es lineal.

In [ ]:
# --- Preparacion de senales (Spearman = correlacion de rangos) ---
def preparar(v):
    v = rankdata(v) if CORR_METHOD == "spearman" else np.asarray(v, float).copy()
    v = v - v.mean()
    return v / (np.linalg.norm(v) + 1e-12)          # norma 1 -> el producto punto es Pearson/Spearman

# Rango de lags comun a todo el bloque
Lb        = int(round(LAG_MAX / BIN))
lags_bins = np.arange(-Lb, Lb + 1)
lags_s    = lags_bins * BIN

def analizar_acople(neural, pupil1d, n_perm=N_PERM):
    # Correlacion cruzada circular (FFT) + nulo por permutacion circular.
    # ESTADISTICO = maximo de |correlacion| en la ventana de lags (el signo de PC es arbitrario).
    a = preparar(neural); b = preparar(pupil1d); N = len(a)
    # ifft(conj(A)*B)[m] = sum_t a[t]*b[t+m]  -> lag m positivo = pupila desplazada al futuro = retrasada.
    C = np.fft.irfft(np.conj(np.fft.rfft(a)) * np.fft.rfft(b), n=N)
    curva = C[(0 + lags_bins) % N]                  # correlacion observada vs lag (con signo)
    i = int(np.argmax(np.abs(curva))); stat = float(abs(curva[i]))
    rloc = np.random.default_rng(SEED)              # nulo reproducible (semilla fija)
    offs = rloc.integers(1, N, size=n_perm)
    null_lag = np.stack([C[(s + lags_bins) % N] for s in offs])   # (n_perm, n_lags) con signo
    stat_null = np.abs(null_lag).max(axis=1)        # max |corr| por permutacion circular
    p = (1 + int(np.sum(stat_null >= stat))) / (n_perm + 1)
    return {"lag": float(lags_s[i]), "r": float(curva[i]), "absr": stat, "p": p,
            "curva": curva, "null_lag": null_lag}

print("Funciones de acoplamiento listas. Lags:", lags_s[0], "..", lags_s[-1], "s")

In [ ]:
# --- Senal neuronal 1D elegida (SENAL_NEURAL) ---
if SENAL_NEURAL == "PC1":
    neural = Y[:, 0].astype(float)
else:
    neural = RATES_z.mean(axis=1).astype(float)     # media de z-scores de la poblacion
pupil1d = PUPILA.astype(float)

ac = analizar_acople(neural, pupil1d)
xcorr_obs, null_lag = ac["curva"], ac["null_lag"]
lag_pico_s, r_pico, stat_obs, p_emp = ac["lag"], ac["r"], ac["absr"], ac["p"]
banda = np.percentile(np.abs(null_lag).max(axis=1), [2.5, 50, 97.5])

print(f"Correlacion ({CORR_METHOD}) | senal = {SENAL_NEURAL}")
signo = "pupila retrasada" if lag_pico_s > 0 else "pupila adelantada" if lag_pico_s < 0 else "lag cero"
print(f"  Lag del pico (|r| max) : {lag_pico_s:+.3f} s  ({signo})")
print(f"  r en el pico           : {r_pico:+.3f}   (|r| = {stat_obs:.3f})")
print(f"  Nulo |r| P2.5/P50/P97.5: {banda[0]:.3f} / {banda[1]:.3f} / {banda[2]:.3f}")
print(f"  p-valor empirico       : {p_emp:.4f}  "
      f"({'SIGNIFICATIVO' if p_emp < 0.05 else 'no significativo'} al 5%)")

### 2.1 Figura central: |correlación cruzada| vs lag + banda del nulo

Esta es **la figura** del proyecto. Dibujamos el **valor absoluto** de la correlación pupila↔estado en
función del retardo (el signo de PC1 no se interpreta); la banda sombreada marca el **techo del nulo**
(P97.5 de |correlación| por lag, respetando la autocorrelación); la línea vertical anota el lag del
pico. Si la curva **supera claramente** la banda, la pupila indexa el estado poblacional con ese
retardo. Si queda **dentro** de la banda, el acoplamiento es indistinguible del azar de dos señales
lentas.

In [ ]:
# Trabajamos con |correlacion| (el signo de PC1 es arbitrario). Banda nula = P97.5 de |nulo| por lag.
abs_obs = np.abs(xcorr_obs)
hi = np.percentile(np.abs(null_lag), 97.5, axis=0)         # techo del nulo, lag a lag
lo = np.zeros_like(hi)

fig, ax = plt.subplots(figsize=(10, 5.5))
ax.fill_between(lags_s, lo, hi, color="gray", alpha=0.30, label="nulo circular (P97.5 de |r|)")
ax.plot(lags_s, abs_obs, color="tab:blue", lw=2, label=f"|correlacion| observada ({CORR_METHOD})")
ax.axvline(lag_pico_s, color="tab:red", ls="--",
           label=f"lag del pico = {lag_pico_s:+.2f} s (|r|={stat_obs:.2f})")
ax.axvline(0, color="black", lw=0.8, alpha=0.5)
ax.set_xlabel("Lag (s)   [+ = pupila retrasada respecto al estado neuronal]")
ax.set_ylabel(f"|Correlacion| ({CORR_METHOD})")
ax.set_title(f"Acoplamiento pupila <-> {SENAL_NEURAL}  |  p_emp = {p_emp:.4f}")
ax.legend(loc="upper right"); plt.tight_layout(); plt.show()

### 2.2 ¿Y si PC1 no es el eje de arousal? Comparación de señales

PC1 solo captura una fracción pequeña de la varianza durante el espontáneo (la población es de alta
dimensión), así que el eje ligado a la pupila **quizá no sea PC1**. Comparamos varias definiciones de
"señal neuronal" —PC1, PC2, PC3 y la **media de z-scores** de toda la población (`mean_z`, un proxy
clásico de *arousal*)— contra la pupila, cada una con su propio nulo circular.

> **Aviso estadístico:** barrer varias señales **infla el riesgo de falso positivo**. El p-valor de
> cada fila es *por señal*, sin corrección por comparaciones múltiples; léelo como **exploratorio**. Si
> quieres reportar una como resultado principal, fíjala en `SENAL_NEURAL` y re-ejecuta el análisis
> confirmatorio de arriba.

In [ ]:
candidatos_senal = {"PC1": Y[:, 0]}
if N_PCA > 1: candidatos_senal["PC2"] = Y[:, 1]
if N_PCA > 2: candidatos_senal["PC3"] = Y[:, 2]
candidatos_senal["mean_z (poblacion)"] = RATES_z.mean(axis=1)

filas = []
for nombre, sig in candidatos_senal.items():
    r = analizar_acople(np.asarray(sig, float), pupil1d)
    filas.append({"senal": nombre, "lag_pico_s": r["lag"], "r_signed": r["r"],
                  "abs_r": r["absr"], "p_emp": r["p"]})
diag = pd.DataFrame(filas).sort_values("abs_r", ascending=False).reset_index(drop=True)
print("Acoplamiento con la pupila por definicion de senal (ordenado por |r|):")
display(diag.round(4))

mejor = diag.iloc[0]
print(f"\nMejor senal: {mejor['senal']}  |r|={mejor['abs_r']:.3f} a lag {mejor['lag_pico_s']:+.2f} s  "
      f"p_emp={mejor['p_emp']:.4f}  ({'significativo' if mejor['p_emp']<0.05 else 'no significativo'})")
print("(p por-senal, exploratorio, sin correccion por comparaciones multiples).")

### 2.3 Control por carrera (el *confound* que exige el SPEC)

*Arousal* (pupila) y **locomoción** (carrera) son disociables, pero covarían mucho, y la carrera domina
la actividad cortical. Si no lo controlamos, no podemos distinguir "**la pupila indexa el cortex**" de
"**ambas siguen a la carrera**". Aquí regresamos la carrera (contemporánea) fuera de la señal neuronal y
de la pupila, y recalculamos la correlación cruzada **sobre los residuos** ("acoplamiento parcial"),
con su propio nulo circular. También reportamos cuánto se acoplan carrera↔estado y carrera↔pupila.

> **Límite del control:** solo quitamos la componente **lineal y contemporánea** de la carrera; efectos
> retardados o no lineales de la locomoción pueden quedar. Es un control razonable, no una prueba de
> mediación causal.

In [ ]:
if CARRERA is not None:
    def quitar_carrera(sig, run):
        # Residuo de 'sig' tras regresion lineal sobre 'run' (con intercepto): elimina la locomocion contemporanea.
        A = np.column_stack([run, np.ones_like(run)])
        beta, *_ = np.linalg.lstsq(A, sig, rcond=None)
        return sig - A @ beta

    run_c      = CARRERA.astype(float)
    neural_res = quitar_carrera(neural, run_c)
    pupil_res  = quitar_carrera(pupil1d, run_c)

    ac_full = ac                                      # pupila<->estado (simple), ya calculado
    ac_part = analizar_acople(neural_res, pupil_res)  # pupila<->estado PARCIAL (sin carrera)
    ac_nr   = analizar_acople(neural,  run_c)         # carrera<->estado
    ac_pr   = analizar_acople(pupil1d, run_c)         # carrera<->pupila

    print("Acoplamientos  (|r|, lag, p):")
    print(f"  pupila <-> {SENAL_NEURAL}  (simple)          : |r|={ac_full['absr']:.3f}  lag={ac_full['lag']:+.2f}s  p={ac_full['p']:.4f}")
    print(f"  pupila <-> {SENAL_NEURAL}  (parcial | carrera): |r|={ac_part['absr']:.3f}  lag={ac_part['lag']:+.2f}s  p={ac_part['p']:.4f}")
    print(f"  carrera <-> {SENAL_NEURAL}                    : |r|={ac_nr['absr']:.3f}  lag={ac_nr['lag']:+.2f}s  p={ac_nr['p']:.4f}")
    print(f"  carrera <-> pupila                            : |r|={ac_pr['absr']:.3f}  lag={ac_pr['lag']:+.2f}s  p={ac_pr['p']:.4f}")

    fig, axx = plt.subplots(figsize=(9, 5))
    axx.plot(lags_s, np.abs(ac_full["curva"]), color="tab:blue", lw=2, label="pupila<->estado (simple)")
    axx.plot(lags_s, np.abs(ac_part["curva"]), color="tab:red",  lw=2, label="parcial (quitada carrera)")
    hi_p = np.percentile(np.abs(ac_part["null_lag"]), 97.5, axis=0)
    axx.fill_between(lags_s, 0, hi_p, color="gray", alpha=0.25, label="nulo parcial (P97.5)")
    axx.axvline(0, color="black", lw=0.7, alpha=0.4)
    axx.set_xlabel("Lag (s)"); axx.set_ylabel(f"|correlacion| ({CORR_METHOD})")
    axx.set_title("Control por carrera: acoplamiento simple vs parcial")
    axx.legend(); plt.tight_layout(); plt.show()
else:
    print("No hay senal de carrera cargada; no se puede controlar por locomocion.")

---
## Bloque 3 — *Decoding* directo con barrido de lag τ (REFUERZO)

Un segundo método, **independiente**, que debería apuntar al mismo retardo. Planteamos un *decoder*
`pupila(t−τ … t) → estado(t)` y **barremos τ** sobre `TAU_GRID`. Comparamos dos modelos:

- **GLM (lineal, baseline):** como el objetivo `Y` es continuo, el GLM *gaussiano* equivale a una
  **regresión lineal regularizada** (`Ridge`, una por componente). Es el modelo más simple e
  interpretable.
- **GRU (no lineal):** recibe una **ventana** de pupila reciente (`DEC_SEQ` bins) y su estado oculto
  "recuerda" el pasado; incorpora implícitamente "leer el pasado reciente".

Como *features* de pupila basta la **pupila suavizada y su derivada** (SPEC §3): doce *features* de
`tsfel` sobre pocos datos serían fragilidad innecesaria.

**Métrica (importante).** En sesiones largas el estado neuronal **deriva** (no-estacionariedad:
electrodo + estado interno). El **R²** castiga cualquier desajuste de media entre train y test, así que
un R² muy negativo puede reflejar *drift*, no que la pupila sea inútil. Por eso reportamos como
**métrica principal la correlación pred↔real** (`r`), que es **robusta al desplazamiento de media**, y
dejamos el R² como secundaria. Además, el interruptor `DETREND_STATE` permite quitar la deriva lineal
(ajustada *solo con train*) para ver el efecto sin el *drift*. Split secuencial, sin barajar.

> **Por qué NO encadenamos decoding + forecasting.** Se descartó la cadena
> "pupila → estado pasado → forecasting → estado actual": propagar hacia delante una reconstrucción
> empobrecida no regenera la dinámica fina perdida; solo compone error. El lag se incorpora **dentro**
> del decoder (τ), no con una etapa intermedia.

### 3.1 *Features* de pupila e infraestructura de muestras

In [ ]:
# --- Features de pupila: suavizada + derivada ---
pupil_sm  = gaussian_filter1d(PUPILA, sigma=SMOOTH_SIGMA / BIN)
pupil_dv  = np.gradient(pupil_sm) / BIN
Xfeat_raw = np.column_stack([pupil_sm, pupil_dv])           # (n_bins, 2)

def detrend_train(M, cut):
    # Quita de cada columna una recta AJUSTADA SOLO CON TRAIN (indices 0..cut). Elimina drift lento.
    x = np.arange(M.shape[0]); out = M.astype(float).copy()
    for j in range(M.shape[1]):
        p = np.polyfit(x[:cut], M[:cut, j], 1)
        out[:, j] = M[:, j] - np.polyval(p, x)
    return out

# Detrend opcional (quita la deriva lineal train-fit de estado y features)
Y_dec = detrend_train(Y, cut) if DETREND_STATE else Y.astype(float)
Xf    = detrend_train(Xfeat_raw, cut) if DETREND_STATE else Xfeat_raw
print(f"DETREND_STATE = {DETREND_STATE}")

# Estandarizar (train-only) para que las metricas sean comparables entre componentes
scaler_X = StandardScaler().fit(Xf[:cut]);    Xfeat = scaler_X.transform(Xf)
scaler_Y = StandardScaler().fit(Y_dec[:cut]); Yz    = scaler_Y.transform(Y_dec)

def r_pred(y_true, y_pred):
    # Correlacion media pred<->real por componente. Robusta al offset de media (a diferencia del R2).
    rs = []
    for j in range(y_true.shape[1]):
        if np.std(y_pred[:, j]) < 1e-9 or np.std(y_true[:, j]) < 1e-9:
            rs.append(0.0)
        else:
            rs.append(np.corrcoef(y_true[:, j], y_pred[:, j])[0, 1])
    return float(np.nanmean(rs))

def indices_tau(n, P, tau, cut):
    # Ancla i (objetivo Y[i]); entrada = ventana de pupila [i-tau-P+1 .. i-tau].
    # Sin fuga: train si objetivo en train (i<cut); test si TODA la entrada en test (i-tau-P+1>=cut).
    tr, te = [], []
    for i in range(n):
        j0, j1 = i - tau - P + 1, i - tau
        if j0 < 0 or j1 < 0 or j1 >= n:
            continue
        if i < cut:          tr.append(i)
        elif j0 >= cut:      te.append(i)
    return np.array(tr, dtype=int), np.array(te, dtype=int)

def plano_tau(Xf, Yt, tau, P, cut):
    tr, te = indices_tau(len(Yt), P, tau, cut)
    feats = lambda idx: np.stack([Xf[i-tau-P+1:i-tau+1].reshape(-1) for i in idx])
    return feats(tr), Yt[tr], feats(te), Yt[te]

def seq_tau(Xf, Yt, tau, P, cut):
    tr, te = indices_tau(len(Yt), P, tau, cut)
    seqs = lambda idx: np.stack([Xf[i-tau-P+1:i-tau+1] for i in idx])
    return seqs(tr), Yt[tr], seqs(te), Yt[te]

def glm_multisalida(Xtr, Ytr, Xte):
    # GLM gaussiano == regresion lineal regularizada; una Ridge por componente.
    preds = [Ridge(alpha=1.0).fit(Xtr, Ytr[:, j]).predict(Xte) for j in range(Ytr.shape[1])]
    return np.stack(preds, axis=1)

class GRURegresor(nn.Module):
    def __init__(self, n_in, n_hidden, n_out):
        super().__init__()
        self.gru = nn.GRU(n_in, n_hidden, batch_first=True)
        self.fc  = nn.Linear(n_hidden, n_out)
    def forward(self, x):
        out, _ = self.gru(x)
        return self.fc(out[:, -1, :])

def entrenar_gru(Xtr, Ytr, Xte, n_in, n_out):
    torch.manual_seed(SEED)
    model = GRURegresor(n_in, GRU_HIDDEN, n_out).to(DEVICE)
    opt   = torch.optim.Adam(model.parameters(), lr=GRU_LR)
    lossf = nn.MSELoss()
    Xt = torch.tensor(Xtr, dtype=torch.float32, device=DEVICE)
    Yt = torch.tensor(Ytr, dtype=torch.float32, device=DEVICE)
    n  = Xt.shape[0]
    model.train()
    for _ in range(GRU_EPOCHS):
        perm = torch.randperm(n, device=DEVICE)             # baraja el ORDEN de muestras, no el tiempo interno
        for k in range(0, n, GRU_BATCH):
            idx = perm[k:k+GRU_BATCH]
            opt.zero_grad()
            loss = lossf(model(Xt[idx]), Yt[idx]); loss.backward(); opt.step()
    model.eval()
    with torch.no_grad():
        return model(torch.tensor(Xte, dtype=torch.float32, device=DEVICE)).cpu().numpy()

print("Infraestructura de decoding lista. Features de pupila:", Xfeat.shape)

### 3.2 Barrido de τ: R² en test para GLM (y GRU)

Para cada τ construimos las muestras, entrenamos y medimos R² (media de las `N_PCA` componentes) en
test. El GLM se calcula siempre; la GRU solo si `RUN_GRU=True` (el SPEC permite recortar a solo GLM).
Convención de τ: **positivo = usar pupila del pasado** para predecir el estado presente (coherente con
que la pupila va retrasada).

In [ ]:
res = []
for tau_s in TAU_GRID:
    tau = int(round(tau_s / BIN))
    # GLM: P=1 (pupila y su derivada en t-tau)
    Xtr, Ytr, Xte, Yte = plano_tau(Xfeat, Yz, tau, P=1, cut=cut)
    if len(Yte) > 5:
        pg = glm_multisalida(Xtr, Ytr, Xte)
        r2_glm, r_glm = r2_score(Yte, pg), r_pred(Yte, pg)
    else:
        r2_glm = r_glm = np.nan
    r2_gru = r_gru = np.nan
    if RUN_GRU:
        Xs, Ys, Xes, Yes = seq_tau(Xfeat, Yz, tau, P=DEC_SEQ, cut=cut)
        if len(Yes) > 5:
            pr = entrenar_gru(Xs, Ys, Xes, n_in=Xfeat.shape[1], n_out=N_PCA)
            r2_gru, r_gru = r2_score(Yes, pr), r_pred(Yes, pr)
    res.append({"tau_s": tau_s, "r_GLM": r_glm, "r_GRU": r_gru,
                "R2_GLM": r2_glm, "R2_GRU": r2_gru})
    print(f"  tau={tau_s:+.2f}s  r_GLM={r_glm:+.3f} R2_GLM={r2_glm:+.3f}"
          + (f" | r_GRU={r_gru:+.3f} R2_GRU={r2_gru:+.3f}" if RUN_GRU else ""))

df_tau = pd.DataFrame(res)
# tau optimo POR CORRELACION (metrica robusta), no por R2
tau_opt_glm = df_tau.loc[df_tau["r_GLM"].idxmax(), "tau_s"]
print(f"\ntau optimo (GLM, por r): {tau_opt_glm:+.2f} s  (lag del pico Bloque 2: {lag_pico_s:+.2f} s)")
if RUN_GRU and df_tau["r_GRU"].notna().any():
    print(f"tau optimo (GRU, por r): {df_tau.loc[df_tau['r_GRU'].idxmax(), 'tau_s']:+.2f} s")
display(df_tau.round(3))

### 3.3 Validación cruzada de métodos: R² vs τ

Si el **τ óptimo del decoding** coincide con el **lag del pico** de la correlación cruzada (Bloque 2),
dos métodos independientes apuntan al mismo número → robustez. Mostramos las **dos métricas**: la
correlación pred↔real (izquierda, robusta al *drift* → la de referencia) y el R² (derecha, sensible al
*drift*: si es muy negativo con `DETREND_STATE=False`, es un síntoma de no-estacionariedad).

In [ ]:
fig, (a1, a2) = plt.subplots(1, 2, figsize=(14, 5))

# (izq) Metrica principal: correlacion pred<->real
a1.plot(df_tau["tau_s"], df_tau["r_GLM"], "o-", color="tab:blue", label="GLM (Ridge)")
if RUN_GRU and df_tau["r_GRU"].notna().any():
    a1.plot(df_tau["tau_s"], df_tau["r_GRU"], "s-", color="tab:orange", label="GRU")
a1.axvline(lag_pico_s, color="tab:red", ls="--", label=f"lag pico Bloque 2 = {lag_pico_s:+.2f} s")
a1.axvline(0, color="black", lw=0.7, alpha=0.4); a1.axhline(0, color="black", lw=0.6, alpha=0.4)
a1.set_xlabel("tau (s)   [+ = pupila del pasado -> estado presente]")
a1.set_ylabel("correlacion pred<->real (media PCs)")
a1.set_title("Decoding: correlacion vs tau  (robusta a drift)"); a1.legend()

# (der) Metrica secundaria: R2 (sensible al drift)
a2.plot(df_tau["tau_s"], df_tau["R2_GLM"], "o-", color="tab:blue", label="GLM")
if RUN_GRU and df_tau["R2_GRU"].notna().any():
    a2.plot(df_tau["tau_s"], df_tau["R2_GRU"], "s-", color="tab:orange", label="GRU")
a2.axvline(lag_pico_s, color="tab:red", ls="--"); a2.axhline(0, color="black", lw=0.6, alpha=0.4)
a2.set_xlabel("tau (s)"); a2.set_ylabel("R2 en test")
a2.set_title("Decoding: R2 vs tau  (sensible a drift)"); a2.legend()
plt.tight_layout(); plt.show()

### 3.4 ¿Es significativo el pico del decoding? Nulo circular

El decoding pica en `r ≈ 0.31`, pero esa magnitud es casi idéntica al |r| **no significativo** del
Bloque 2 → conviene un p-valor propio. Construimos un **nulo circular del decoding**: desplazamos
cíclicamente las *features* de pupila (rompiendo su relación temporal con el estado, preservando su
autocorrelación), re-entrenamos el **GLM** barriendo τ y guardamos el `r` máximo. El estadístico
observado es el `r` máximo real; el p-valor, la fracción del nulo que lo iguala o supera. (Nulo solo
para el GLM, que es el baseline justo; la GRU es demasiado cara para permutar 500 veces.)

In [ ]:
def r_glm_maxtau(Xf, Yt, cut):
    # r maximo (pred<->real) del GLM sobre todo el barrido de tau.
    best = -np.inf
    for tau_s in TAU_GRID:
        tau = int(round(tau_s / BIN))
        Xtr, Ytr, Xte, Yte = plano_tau(Xf, Yt, tau, P=1, cut=cut)
        if len(Yte) > 5:
            best = max(best, r_pred(Yte, glm_multisalida(Xtr, Ytr, Xte)))
    return best

N_PERM_DEC = 500
obs_r = float(df_tau["r_GLM"].max())
rloc  = np.random.default_rng(SEED)
null_r = np.empty(N_PERM_DEC)
for k in range(N_PERM_DEC):
    off = int(rloc.integers(1, len(Yz)))
    null_r[k] = r_glm_maxtau(np.roll(Xfeat, off, axis=0), Yz, cut)   # pupila desplazada ciclicamente
p_dec = (1 + int(np.sum(null_r >= obs_r))) / (N_PERM_DEC + 1)

print(f"Decoding GLM: r_max observado = {obs_r:.3f}")
print(f"Nulo circular ({N_PERM_DEC} perm): P50={np.percentile(null_r,50):.3f}  "
      f"P95={np.percentile(null_r,95):.3f}  P97.5={np.percentile(null_r,97.5):.3f}")
print(f"p-valor empirico = {p_dec:.4f}  "
      f"({'SIGNIFICATIVO' if p_dec < 0.05 else 'no significativo'} al 5%)")

### 3.5 Confirmación del *drift*: R² con y sin *detrend*

Comprobamos la hipótesis de que el R²<0 era **no-estacionariedad**, no ausencia de señal. Evaluamos el
GLM al τ óptimo **quitando** la deriva lineal (train-fit) y **sin** quitarla. Esperado: al hacer
*detrend* el **R² sube hacia ~0** mientras la **correlación se mantiene ~0.3** → el R² negativo era drift.

In [ ]:
Xf_base = np.column_stack([pupil_sm, pupil_dv])
tau     = int(round(tau_opt_glm / BIN))
comp = []
for det in [False, True]:
    Ydec = detrend_train(Y, cut)        if det else Y.astype(float)
    Xf_r = detrend_train(Xf_base, cut)  if det else Xf_base
    Xf_z = StandardScaler().fit(Xf_r[:cut]).transform(Xf_r)
    Yz_d = StandardScaler().fit(Ydec[:cut]).transform(Ydec)
    Xtr, Ytr, Xte, Yte = plano_tau(Xf_z, Yz_d, tau, P=1, cut=cut)
    pg = glm_multisalida(Xtr, Ytr, Xte)
    comp.append({"DETREND": det, "R2": r2_score(Yte, pg), "r": r_pred(Yte, pg)})
display(pd.DataFrame(comp).round(3))
print(f"(GLM al tau optimo = {tau_opt_glm:+.2f} s)")
print("Si el R2 sube hacia ~0 con detrend y la r se mantiene -> el R2<0 era DRIFT, no ausencia de senal.")

---
## Bloque 4 — Contraste: *forecasting* intrínseco del estado (OPCIONAL)

Pieza **independiente** y claramente etiquetada; es lo primero que se elimina si falta tiempo
(`RUN_BLOQUE4=False`). Hacemos *forecasting* autorregresivo puro `estado(t−P … t) → estado(t+h)`
a varios horizontes `h`, **sin pupila**.

**Por qué contextualiza el límite:** el estado completo predice bastante mejor su propio futuro que lo
que la pupila reconstruye del presente. Esa **brecha** hace visible que la actividad poblacional es
**multidimensional** y la pupila solo ve su **eje lento de arousal**: es un índice **parcial**.

In [ ]:
if RUN_BLOQUE4:
    def indices_ar(n, P, h, cut):
        tr, te = [], []
        for i in range(P - 1, n - h):
            if   i + h < cut:      tr.append(i)
            elif i - P + 1 >= cut: te.append(i)
        return np.array(tr, dtype=int), np.array(te, dtype=int)

    r2_ar = []
    for h in AR_HORIZONTES:
        tr, te = indices_ar(n_bins, AR_LAGS, h, cut)
        feats = lambda idx: np.stack([Yz[i-AR_LAGS+1:i+1].reshape(-1) for i in idx])
        Xtr, Xte = feats(tr), feats(te)
        Ytr, Yte = Yz[tr + h], Yz[te + h]
        pred = glm_multisalida(Xtr, Ytr, Xte)
        r2_ar.append(r2_score(Yte, pred))
        print(f"  horizonte h={h} bins ({h*BIN*1000:.0f} ms):  R2 = {r2_ar[-1]:.3f}")
    df_ar = pd.DataFrame({"h_bins": AR_HORIZONTES,
                          "h_ms": [h*BIN*1000 for h in AR_HORIZONTES], "R2_AR": r2_ar})
    display(df_ar.round(3))
else:
    df_ar = None
    print("Bloque 4 desactivado (RUN_BLOQUE4=False).")

---
## Bloque 5 — Relato, figuras y límites declarados (ÚLTIMA HORA)

El relato y los límites valen tanto como el análisis. Reunimos las figuras en orden narrativo y cerramos
con conclusiones y las salvaguardas del proyecto.

**Narrativa:**
1. Pupila y estado poblacional están **acopladas**, con la pupila retrasada ~X s *(Bloque 2)*.
2. Por eso se puede **reconstruir** el estado desde la pupila; R² máximo a ese lag *(Bloque 3)*.
3. Pero la pupila es **unidimensional** (arousal) y captura solo una fracción; el *forecasting* desde el
   estado completo predice mucho mejor *(Bloque 4, si se hizo)*.

In [ ]:
n_paneles = 3 if (RUN_BLOQUE4 and df_ar is not None) else 2
fig, axes = plt.subplots(1, n_paneles, figsize=(6*n_paneles, 5))

# (1) |Correlacion cruzada| + nulo
ax = axes[0]
ax.fill_between(lags_s, lo, hi, color="gray", alpha=0.30)
ax.plot(lags_s, abs_obs, color="tab:blue", lw=2)
ax.axvline(lag_pico_s, color="tab:red", ls="--")
ax.set_title(f"1) Acoplamiento pupila<->estado\nlag pico {lag_pico_s:+.2f}s, p={p_emp:.4f}")
ax.set_xlabel("lag (s)"); ax.set_ylabel(f"|corr| ({CORR_METHOD})")

# (2) correlacion pred<->real vs tau (metrica robusta)
ax = axes[1]
ax.plot(df_tau["tau_s"], df_tau["r_GLM"], "o-", color="tab:blue", label="GLM")
if RUN_GRU and df_tau["r_GRU"].notna().any():
    ax.plot(df_tau["tau_s"], df_tau["r_GRU"], "s-", color="tab:orange", label="GRU")
ax.axvline(lag_pico_s, color="tab:red", ls="--"); ax.axhline(0, color="black", lw=0.6, alpha=0.4)
ax.set_title("2) Reconstruccion del estado\nr(pred,real) vs lag tau"); ax.set_xlabel("tau (s)")
ax.set_ylabel("r pred<->real"); ax.legend()

# (3) Forecasting intrinseco (R2) vs mejor decoding de pupila (r)
if n_paneles == 3:
    ax = axes[2]
    ax.plot(df_ar["h_ms"], df_ar["R2_AR"], "^-", color="tab:green", label="forecasting AR del estado (R2)")
    ax.axhline(0, color="black", lw=0.6, alpha=0.4)
    best_r = np.nanmax([df_tau["r_GLM"].max(),
                        df_tau["r_GRU"].max() if RUN_GRU else np.nan])
    ax.set_title(f"3) Indice parcial\nmejor decoding pupila: r={best_r:.2f}")
    ax.set_xlabel("horizonte (ms)"); ax.set_ylabel("R2 forecasting AR"); ax.legend()

plt.tight_layout(); plt.show()

### 5.1 Conclusiones y límites declarados

**Conclusión (elegir la rama según los números obtenidos):**

- **Si el pico de |correlación| supera el nulo (p < 0.05):** la pupila indexa el estado poblacional de
  VISp en espontáneo, con un retardo de ~**`lag_pico_s`** s; el *decoding* lo confirma (τ óptimo, por
  `r`, ≈ lag del pico) y reconstruye el estado con **r ≈ `best_r`**. El *forecasting* intrínseco predice
  mejor → la pupila es un índice **parcial**, ligado al eje lento de *arousal*.
- **Si el pico queda dentro del nulo (p ≥ 0.05), como ocurre con PC1 en esta sesión:** el acoplamiento
  pupila↔PC1 **no es distinguible** de lo que producen dos señales lentas por azar. Esto **no** es un
  fracaso del método —el nulo circular nos protege de sobre-interpretar— sino un resultado: en esta
  sesión, con PC1 como resumen, la pupila no es un índice fiable del estado. La tabla del §2.2 dice si
  **otra señal** (p. ej. `mean_z`) captura mejor el eje de *arousal*, y `DETREND_STATE` dice si la
  no-estacionariedad estaba enmascarando el efecto en el *decoding*.

**Límites (protegen el proyecto):**
- Es **una sola sesión**: todo es **descriptivo**, no generaliza a una población de animales.
- La lectura "pupila = arousal" es válida **porque** trabajamos bajo **luminancia constante** (el gris
  del bloque espontáneo); esto justifica *a posteriori* elegir el régimen espontáneo frente a los
  *gratings*.
- El nulo circular controla la autocorrelación, pero no descarta que **una tercera variable**
  (p. ej. la carrera) module ambas señales; por eso registramos la carrera y conviene reportar la
  correlación pupila↔estado **controlando por carrera** como extensión.

In [ ]:
# Buenas practicas: cerrar la conexion de streaming para liberar recursos (importante en Windows).
io.close(); h5_file.close(); rem_file.close()
print("Archivo NWB cerrado. Recursos liberados.")